In [ ]:
#Cell 1	GPU Setup	10 sec	Detects GPU, enables mixed precision

#run this cell when restarted
import tensorflow as tf
import os

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f'GPU Detected: {gpus[0].name}')
    except RuntimeError as e:
        print(e)
else:
    print('No GPU detected. Please enable GPU in Runtime > Change runtime type')

tf.keras.mixed_precision.set_global_policy('mixed_float16')
print(f'TensorFlow version: {tf.__version__}')
print(f'GPU Available: {tf.test.is_gpu_available()}')
print(f'Built with CUDA: {tf.test.is_built_with_cuda()}')

Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.


GPU Detected: /physical_device:GPU:0
TensorFlow version: 2.19.0
GPU Available: True
Built with CUDA: True


In [ ]:
#Cell 2	Mount Google Drive	20 sec	Connects to your Drive
#run this cell when restarted
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#Cell 3	Extract Dataset ZIP	3-5 min	Extracts videos (skip if done)
import zipfile
import os

zip_path = '/content/drive/MyDrive/Real Life Violence Dataset.zip'
extract_path = '/content/RLVS'

if os.path.exists(extract_path) and len(os.listdir(extract_path)) > 0:
    print('Dataset already extracted. Skipping extraction.')
else:
    print('Extracting dataset...')
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content')
    print('Extraction complete.')

dataset_path = '/content/Real Life Violence Dataset'
violence_path = os.path.join(dataset_path, 'Violence')
nonviolence_path = os.path.join(dataset_path, 'NonViolence')

print(f'Violence videos: {len(os.listdir(violence_path))}')
print(f'NonViolence videos: {len(os.listdir(nonviolence_path))}')

Extracting dataset...
Extraction complete.
Violence videos: 1000
NonViolence videos: 1000


In [ ]:
#Cell 4	Extract Frames	30-45 min	Creates 24K frames (skip if done)
import cv2
import numpy as np
from tqdm import tqdm
import os

# Smart path detection - use Drive if available, otherwise local
drive_frames_path = '/content/drive/MyDrive/Violence_Detection_Frames'
local_frames_path = '/content/processed_frames'

if os.path.exists(drive_frames_path) and len(os.listdir(drive_frames_path)) > 0:
    processed_frames_path = drive_frames_path
    print('✓ Using frames from Google Drive')
    print(f'Location: {drive_frames_path}')
else:
    processed_frames_path = local_frames_path
    print('Using frames from local storage')

violence_frames_path = os.path.join(processed_frames_path, 'Violence')
nonviolence_frames_path = os.path.join(processed_frames_path, 'NonViolence')

os.makedirs(violence_frames_path, exist_ok=True)
os.makedirs(nonviolence_frames_path, exist_ok=True)

def extract_frames(video_path, output_folder, num_frames=12):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < num_frames:
        cap.release()
        return False

    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)

    cap.release()

    if len(frames) == num_frames:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        for i, frame in enumerate(frames):
            frame_path = os.path.join(output_folder, f'{video_name}_frame_{i}.jpg')
            cv2.imwrite(frame_path, frame)
        return True
    return False

violence_frames_exist = len(os.listdir(violence_frames_path)) > 0
nonviolence_frames_exist = len(os.listdir(nonviolence_frames_path)) > 0

if violence_frames_exist and nonviolence_frames_exist:
    print('✓ Frames already extracted. Skipping frame extraction.')
    print(f'Violence frames: {len(os.listdir(violence_frames_path))}')
    print(f'NonViolence frames: {len(os.listdir(nonviolence_frames_path))}')
else:
    print('Extracting frames from Violence videos...')
    violence_videos = [os.path.join(violence_path, v) for v in os.listdir(violence_path)]
    for video in tqdm(violence_videos, desc='Violence'):
        extract_frames(video, violence_frames_path, num_frames=12)

    print('Extracting frames from NonViolence videos...')
    nonviolence_videos = [os.path.join(nonviolence_path, v) for v in os.listdir(nonviolence_path)]
    for video in tqdm(nonviolence_videos, desc='NonViolence'):
        extract_frames(video, nonviolence_frames_path, num_frames=12)

    print('Frame extraction complete.')
    print(f'Violence frames: {len(os.listdir(violence_frames_path))}')
    print(f'NonViolence frames: {len(os.listdir(nonviolence_frames_path))}')


✓ Using frames from Google Drive
Location: /content/drive/MyDrive/Violence_Detection_Frames
✓ Frames already extracted. Skipping frame extraction.
Violence frames: 11988
NonViolence frames: 11964


In [ ]:
#Cell 4a
#old cell 4 do not run this cell


import cv2
import numpy as np
from tqdm import tqdm
import os

processed_frames_path = '/content/processed_frames'
violence_frames_path = os.path.join(processed_frames_path, 'Violence')
nonviolence_frames_path = os.path.join(processed_frames_path, 'NonViolence')

os.makedirs(violence_frames_path, exist_ok=True)
os.makedirs(nonviolence_frames_path, exist_ok=True)

def extract_frames(video_path, output_folder, num_frames=12):
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < num_frames:
        cap.release()
        return False

    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)
    frames = []

    for idx in frame_indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            frame = cv2.resize(frame, (224, 224))
            frames.append(frame)

    cap.release()

    if len(frames) == num_frames:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        for i, frame in enumerate(frames):
            frame_path = os.path.join(output_folder, f'{video_name}_frame_{i}.jpg')
            cv2.imwrite(frame_path, frame)
        return True
    return False

violence_frames_exist = len(os.listdir(violence_frames_path)) > 0
nonviolence_frames_exist = len(os.listdir(nonviolence_frames_path)) > 0

if violence_frames_exist and nonviolence_frames_exist:
    print('Frames already extracted. Skipping frame extraction.')
else:
    print('Extracting frames from Violence videos...')
    violence_videos = [os.path.join(violence_path, v) for v in os.listdir(violence_path)]
    for video in tqdm(violence_videos, desc='Violence'):
        extract_frames(video, violence_frames_path, num_frames=12)

    print('Extracting frames from NonViolence videos...')
    nonviolence_videos = [os.path.join(nonviolence_path, v) for v in os.listdir(nonviolence_path)]
    for video in tqdm(nonviolence_videos, desc='NonViolence'):
        extract_frames(video, nonviolence_frames_path, num_frames=12)

    print('Frame extraction complete.')

print(f'Violence frames: {len(os.listdir(violence_frames_path))}')
print(f'NonViolence frames: {len(os.listdir(nonviolence_frames_path))}')

Extracting frames from Violence videos...


Violence: 100%|██████████| 1000/1000 [19:01<00:00,  1.14s/it]


Extracting frames from NonViolence videos...


NonViolence: 100%|██████████| 1000/1000 [07:16<00:00,  2.29it/s]

Frame extraction complete.
Violence frames: 11988
NonViolence frames: 11964


In [ ]:
#Cell 4b
#old cell 4a do not run this cell


import shutil
import os

# Path to save frames in Google Drive
drive_frames_path = '/content/drive/MyDrive/Violence_Detection_Frames'

# Check if already copied
if os.path.exists(drive_frames_path) and len(os.listdir(drive_frames_path)) > 0:
    print('Frames already exist in Google Drive. Skipping copy.')
    print(f'Location: {drive_frames_path}')
else:
    print('Copying frames to Google Drive...')
    print('This will take 5-10 minutes for 24,000 frames...')

    shutil.copytree('/content/processed_frames', drive_frames_path)

    print('✓ Frames successfully saved to Google Drive!')
    print(f'Location: {drive_frames_path}')
    print(f'Violence frames: {len(os.listdir(os.path.join(drive_frames_path, "Violence")))}')
    print(f'NonViolence frames: {len(os.listdir(os.path.join(drive_frames_path, "NonViolence")))}')


Copying frames to Google Drive...
This will take 5-10 minutes for 24,000 frames...
✓ Frames successfully saved to Google Drive!
Location: /content/drive/MyDrive/Violence_Detection_Frames
Violence frames: 11988
NonViolence frames: 11964


In [ ]:
#Cell 5	Prepare Data Split	2 min	Creates train/test split
#one go cells
import tensorflow as tf
from sklearn.model_selection import train_test_split
import numpy as np
import os

def load_image_paths_and_labels():
    image_paths = []
    labels = []

    for img in os.listdir(violence_frames_path):
        image_paths.append(os.path.join(violence_frames_path, img))
        labels.append(1)

    for img in os.listdir(nonviolence_frames_path):
        image_paths.append(os.path.join(nonviolence_frames_path, img))
        labels.append(0)

    return np.array(image_paths), np.array(labels)

image_paths, labels = load_image_paths_and_labels()
print(f'Total images: {len(image_paths)}')
print(f'Violence: {np.sum(labels == 1)}, NonViolence: {np.sum(labels == 0)}')

X_train, X_test, y_train, y_test = train_test_split(
    image_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

y_train = tf.keras.utils.to_categorical(y_train, 2)
y_test = tf.keras.utils.to_categorical(y_test, 2)

print(f'Train samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

Total images: 23952
Violence: 11988, NonViolence: 11964
Train samples: 19161
Test samples: 4791
